# blueprint-vlm — train on a rented GPU
Runtime → Change runtime type → **A100** (or any 24GB+ card).
Order matters: baseline first, so the delta is attributable.

In [ ]:
!git clone https://github.com/USERNAME/blueprint-vlm.git
%cd blueprint-vlm
!pip install -q -r requirements.txt

In [ ]:
!python scripts/build_dataset.py --out data/synth

## 1. Zero-shot baseline — before any training

In [ ]:
!python -m blueprint_vlm.infer --config configs/qwen25vl_3b_lora.yaml \
    --split data/synth/test --out results/zeroshot_test.json
!python scripts/score.py --split data/synth/test \
    --predictions results/zeroshot_test.json \
    --out results/zeroshot_test_scored.json --title 'zero-shot / test'

## 2. Fine-tune

In [ ]:
!python -m blueprint_vlm.train --config configs/qwen25vl_3b_lora.yaml

## 3. Score both held-out sets

In [ ]:
for split in ['test', 'test_hard']:
    !python -m blueprint_vlm.infer --config configs/qwen25vl_3b_lora.yaml \
        --split data/synth/{split} --adapter outputs/qwen25vl-3b-lora \
        --out results/finetuned_{split}.json
    !python scripts/score.py --split data/synth/{split} \
        --predictions results/finetuned_{split}.json \
        --out results/finetuned_{split}_scored.json --title 'fine-tuned / {split}'

## 4. Resolution ablation
`max_pixels` moves the counting tasks more than the learning rate does.

In [ ]:
import yaml, pathlib
base = yaml.safe_load(open('configs/qwen25vl_3b_lora.yaml'))
for mp in [401408, 802816, 1204224]:
    cfg = dict(base, max_pixels=mp)
    p = f'configs/_abl_{mp}.yaml'; yaml.safe_dump(cfg, open(p,'w'))
    !python -m blueprint_vlm.infer --config {p} --split data/synth/test \
        --adapter outputs/qwen25vl-3b-lora --out results/abl_mp{mp}.json
    !python scripts/score.py --split data/synth/test \
        --predictions results/abl_mp{mp}.json --title 'max_pixels={mp}'